# Notebook 21 — Transformer Baseline (Fine-tuned RoBERTa) with Full Fairness Audit

**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

Adds a modern transformer as a sixth model, fine-tuned on the **same** TweetEval train
split and evaluated on the **same** test set, subgroups, and fairness metrics as the five
classical baselines. This lets the thesis ask a deeper question:

> Do the linguistic-subgroup fairness gaps found in classical models **persist in a
> fine-tuned transformer**, or does a stronger model remove them?

Design (so it is a fair peer, not a parallel study):
- Base model: `cardiffnlp/twitter-roberta-base-sentiment-latest` (RoBERTa pre-trained on
  tweet sentiment; same 3-class scheme negative/neutral/positive).
- Fine-tuned on `tw_train`; evaluated on `tw_test` with the identical `subgroup_primary`
  tags and SEED=42.
- Predictions saved in the standard schema (`y_true,y_pred,proba_*,confidence,correct,
  subgroup`) so NB17R, the AIF360+Fairlearn audit, per-class, HCER and ECE cells pick it
  up as "Model 6" with no changes.

**GPU required.** Runtime -> Change runtime type -> GPU (T4 fine). Training is
checkpointed to Drive so a disconnect does not lose the run.


## Cell 1: Setup and GPU check

In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas pyarrow
from google.colab import drive; drive.mount('/content/drive')
import sys; sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *
import pandas as pd, numpy as np, torch, os
print("CUDA available:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> GPU"

CKPT_DIR = PATHS["results"].parent / "roberta_ckpt"   # on Drive, survives disconnects
CKPT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
CLASSES=["negative","neutral","positive"]
LABEL2ID={c:i for i,c in enumerate(CLASSES)}; ID2LABEL={i:c for c,i in LABEL2ID.items()}
print("checkpoint dir:", CKPT_DIR)

Mounted at /content/drive
thesis_utils loaded. [V2 FIXED: Fairlearn EOD, Theil index]
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal
CUDA available: True | device: Tesla T4
checkpoint dir: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/roberta_ckpt


## Cell 2: Load the same train/test split + subgroup tags

In [2]:
D=PATHS["data"]
tw_train=pd.read_parquet(D/"tw_train.parquet").reset_index(drop=True)
tw_test =pd.read_parquet(D/"tw_test.parquet").reset_index(drop=True)

# text column: prefer raw text (transformers do their own tokenisation), fall back to clean
TXT = "text" if "text" in tw_train.columns else "text_clean"
print("using text column:", TXT)

# map string sentiment -> id
def to_id(s): return LABEL2ID[s]
tw_train["label"]=tw_train["sentiment"].map(to_id)
tw_test["label"] =tw_test["sentiment"].map(to_id)

# subgroup tags for the test set (same 4-subgroup scheme)
if "subgroup_primary" in tw_test.columns:
    sg_test=tw_test["subgroup_primary"].values
else:
    tw_test=tag_subgroups(tw_test, text_col=TXT); sg_test=tw_test["subgroup_primary"].values

print("train", len(tw_train), "test", len(tw_test))
print("label dist (train):", tw_train["label"].value_counts().to_dict())

using text column: text
train 45615 test 12284
label dist (train): {1: 20673, 2: 17849, 0: 7093}


## Cell 3: Tokenise

In [3]:
from transformers import AutoTokenizer
BASE="cardiffnlp/twitter-roberta-base-sentiment-latest"
tok=AutoTokenizer.from_pretrained(BASE)

from datasets import Dataset
def mk(ds):
    d=Dataset.from_pandas(ds[[TXT,"label"]].rename(columns={TXT:"text"}))
    return d.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)
train_ds=mk(tw_train); test_ds=mk(tw_test)
print("tokenised.")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Map:   0%|          | 0/45615 [00:00<?, ? examples/s]

Map:   0%|          | 0/12284 [00:00<?, ? examples/s]

tokenised.


## Cell 4: Fine-tune (checkpointed to Drive)

3 epochs, class-weighted loss to respect the imbalance (negative is the minority),
evaluation each epoch. If a checkpoint already exists the trainer resumes from it, so a
Colab disconnect does not restart training from zero.

In [4]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch.nn as nn

model=AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID, ignore_mismatched_sizes=True)

# class weights (inverse frequency) to counter imbalance
counts=tw_train["label"].value_counts().sort_index().values
w=torch.tensor((counts.sum()/(len(counts)*counts)), dtype=torch.float).to("cuda")
print("class weights:", w.tolist())

class WTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels=inputs.pop("labels")
        out=model(**inputs); logits=out.logits
        loss=nn.CrossEntropyLoss(weight=w)(logits, labels)
        return (loss, out) if return_outputs else loss

from sklearn.metrics import f1_score, accuracy_score
def metrics(p):
    preds=p.predictions.argmax(-1)
    return {"acc":accuracy_score(p.label_ids,preds),
            "macro_f1":f1_score(p.label_ids,preds,average="macro")}

args=TrainingArguments(
    output_dir=str(CKPT_DIR), seed=SEED,
    num_train_epochs=3, per_device_train_batch_size=32, per_device_eval_batch_size=64,
    learning_rate=2e-5, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    logging_steps=100, report_to="none", fp16=True)

trainer=WTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
                 compute_metrics=metrics, tokenizer=tok)

resume = any(p.name.startswith("checkpoint-") for p in CKPT_DIR.iterdir()) if CKPT_DIR.exists() else False
trainer.train(resume_from_checkpoint=resume)
print("training done. best macro F1:", trainer.state.best_metric)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

class weights: [2.143662691116333, 0.7355003952980042, 0.8518684506416321]


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

## Cell 5: Predict on test set and SAVE in the standard schema (becomes Model 6)

In [ ]:
import scipy.special as sp
pred_out=trainer.predict(test_ds)
logits=pred_out.predictions
proba=sp.softmax(logits, axis=1)
y_pred=[ID2LABEL[i] for i in proba.argmax(1)]
y_true=tw_test["sentiment"].values

# save exactly like every other experiment so downstream notebooks pick it up
save_predictions("exp_transformer", "RoBERTa", y_true, y_pred, proba, sg_test)

from sklearn.metrics import classification_report, f1_score, accuracy_score
print("RoBERTa — overall")
print("  Accuracy :", round(accuracy_score(y_true,y_pred),4))
print("  Macro F1 :", round(f1_score(y_true,y_pred,average='macro'),4))
print(classification_report(y_true,y_pred,digits=4))
# also save the fine-tuned model
trainer.save_model(str(CKPT_DIR/"final")); tok.save_pretrained(str(CKPT_DIR/"final"))
print("model saved ->", CKPT_DIR/"final")

## Cell 6: Per-class and per-subgroup — same tables as the classical models

Produces the RoBERTa row for Table 1 (with per-class neg/neu/pos) and the subgroup
breakdown for Table 4, so the transformer sits directly beside the five baselines.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
pred=load_predictions("exp_transformer","RoBERTa")

# Table-1-style row + per class
p,r,f,s=precision_recall_fscore_support(pred["y_true"],pred["y_pred"],labels=CLASSES,average=None,zero_division=0)
row={"Model":"RoBERTa (fine-tuned)","Accuracy":round((pred['y_true']==pred['y_pred']).mean(),4),
     "Macro F1":round(f1_score(pred['y_true'],pred['y_pred'],average='macro'),4)}
for i,c in enumerate(CLASSES):
    row[f"{c.capitalize()} Precision"]=round(p[i],4); row[f"{c.capitalize()} Recall"]=round(r[i],4); row[f"{c.capitalize()} F1"]=round(f[i],4)
print("TABLE 1 ROW (RoBERTa):"); [print(f"  {k}: {v}") for k,v in row.items()]

# Table-4-style subgroup rows
print("\nTABLE 4 (RoBERTa per subgroup):")
sub_rows=[]
for sg in ["emoji-heavy","slang-heavy","sarcasm","formal"]:
    part=pred[pred["subgroup"]==sg]
    if len(part)==0: continue
    yt,yp=part["y_true"].values,part["y_pred"].values
    mf=f1_score(yt,yp,average="macro",zero_division=0)
    lo,hi=bootstrap_ci(yt,yp,n_boot=1000)[1:]
    sub_rows.append({"Subgroup":sg,"N":len(part),"Accuracy":round((yt==yp).mean(),4),
                     "Macro F1":round(mf,4),"CI Low":round(lo,4),"CI High":round(hi,4)})
sub_df=pd.DataFrame(sub_rows)
ref=sub_df[sub_df["Subgroup"]=="formal"]["Macro F1"].iloc[0]
sub_df["F1 Gap vs formal"]=(sub_df["Macro F1"]-ref).round(4)
print(sub_df.to_string(index=False))
sub_df.to_csv(PATHS["results"]/"assembled"/"TableT_RoBERTa_subgroups.csv",index=False)
print("\nKEY: compare these F1 gaps to the LR gaps (Table 4). Do the subgroup gaps persist in the transformer?")

## Cell 7: Full fairness audit (AIF360 + Fairlearn) — RoBERTa

Runs the identical dual-framework audit used for the final model, so Table 9 can include
a RoBERTa panel. Answers: does the transformer pass the fairness thresholds the classical
model failed (sarcasm DIR)?

In [ ]:
pred=load_predictions("exp_transformer","RoBERTa")
audit=full_fairness_audit(pred)
print("RoBERTa — dual-framework audit"); print(audit.to_string(index=False))
audit.to_csv(PATHS["results"]/"assembled"/"TableT_RoBERTa_fairness.csv",index=False)

# HCER / confidence (Table 10 style)
def hcer_block(df, thr=0.70):
    out=[]
    for sg in ["emoji-heavy","slang-heavy","sarcasm","formal","other"]:
        part=df[df["subgroup"]==sg]
        if len(part)==0: continue
        err=part[part["correct"]==0]
        hc=(err["confidence"]>thr).mean() if len(err) else 0
        out.append({"Subgroup":sg,"N miscl":len(err),"HCER":round(hc,4),
                    "MCE":round(err["confidence"].mean(),4) if len(err) else 0})
    return pd.DataFrame(out)
print("\nRoBERTa — confidence (Table 10 style):")
print(hcer_block(pred).to_string(index=False))

## Cell 8: Done — the deeper finding

RoBERTa is now saved as `exp_transformer` in the standard schema. Re-run NB17R to pull it
into the consolidated tables, and the AIF360+Fairlearn audit already produced its panel.

The headline research question to write up: **whether the subgroup fairness gaps and the
confident-error pattern persist in a fine-tuned transformer.** If they do, the thesis
claim strengthens from "classical models are biased" to "these linguistic-subgroup harms
are not solved by moving to a modern pretrained model." If the transformer closes them,
that is an equally interesting, publishable finding.

In [ ]:
print("NB21 complete. RoBERTa saved as exp_transformer — re-run NB17R to consolidate.")